# 05 — Honest reconstruction with Airflow backfills

Airflow's backfill machinery (`catchup=True`) is built for reconstruction:
re-run a DAG once per logical date and rebuild history partition by
partition. The problem: a naive backfill against vacuumed sources either
**fails at runtime** or — worse — **silently writes zeros** that look like
real facts.

With alethe, every backfill run is gated by the PIT report:

| Zone at logical date | Backfill behaviour |
|---|---|
| **CERTAIN** | rebuild exactly — sources time-travel to the logical date |
| **BOUNDED** | rebuild, but stamp `pit_verdict='BOUNDED'` on every row — the numbers are lower bounds |
| **UNACHIEVABLE** | `AirflowSkipException` — visibly absent in the grid, never fabricated |

The Airflow grid becomes an **honesty map** of the reconstruction.

This notebook executes the gating logic for real against a Delta table
(section 2–4), then shows the production DAG (section 5).

In [1]:
import shutil, time
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd
from deltalake import DeltaTable, write_deltalake
import alethe

WS = Path("reconstruct_workspace")
if WS.exists():
    shutil.rmtree(WS)
WS.mkdir()
print(f"alethe {alethe.__version__}  →  {WS.resolve()}")

alethe 0.1.0  →  /Users/seamusaran/Documents/alethe/notebooks/reconstruct_workspace


## 1. A source with partially destroyed history

Six versions of `orders` written ~1.5s apart (each second stands in for a
day of history), then a real `VACUUM`. The oracle will find the boundary.

In [2]:
ORDERS = WS / "orders"

for v in range(6):
    df = pd.DataFrame({
        "customer_id": [f"C{i % 3}" for i in range(6)],
        "amount":      [100.0 * (v + 1) + i for i in range(6)],
    })
    write_deltalake(ORDERS, df, mode="overwrite")
    time.sleep(1.5)

removed = DeltaTable(str(ORDERS)).vacuum(
    retention_hours=0, enforce_retention_duration=False, dry_run=False)
print(f"6 versions written, {len(removed)} parquet files vacuumed")

wm = alethe.watermark(ORDERS)
alethe.record(wm, WS / "watermarks.jsonl")
print(f"{wm.chain}: boundary v{wm.boundary['version']}, "
      f"validated={wm.empirically_validated}")
print(f"  earliest {wm.earliest_dt.strftime('%H:%M:%S')} → "
      f"boundary {wm.boundary_dt.strftime('%H:%M:%S')}")

6 versions written, 5 parquet files vacuumed
delta://orders: boundary v5, validated=True
  earliest 15:48:48 → boundary 15:48:55


## 2. The backfill window

Nine logical dates spanning all three zones: before the table existed,
inside the vacuumed window, and inside retention.

In [3]:
report = alethe.pit_report("reporting.revenue_daily", [wm])
print(report)

span = report.effective_boundary - report.earliest_dt
logical_dates = (
    [report.earliest_dt - timedelta(seconds=s) for s in (20, 10)]      # UNACHIEVABLE
    + [report.earliest_dt + span * f for f in (0.15, 0.4, 0.65, 0.9)]  # BOUNDED
    + [report.effective_boundary + timedelta(seconds=s) for s in (0, 2, 4)]  # CERTAIN
)

PIT Achievability Report: reporting.revenue_daily
────────────────────────────────────────────────────────
Upstream chain                       Boundary                 Grade
  delta://orders                     2026-07-04 15:48:55      EvidenceGrade.DERIVED ← limiting
────────────────────────────────────────────────────────
Effective boundary:  2026-07-04 15:48:55  (limiting: delta://orders)
Effective grade:     EvidenceGrade.DERIVED

PIT zones:
  CERTAIN        2026-07-04 15:48:55 → now
  BOUNDED        2026-07-04 15:48:48 → 2026-07-04 15:48:55  limiting: ['delta://orders']
  UNACHIEVABLE   −∞ → 2026-07-04 15:48:48  limiting: ['delta://orders']


## 3. The gated backfill loop

This is exactly what the Airflow task does, executed inline. The PIT read
is real Delta time travel (`load_as_version(timestamp)`); the aggregation
stands in for the rewritten model SQL.

In [4]:
from alethe import PitStatus

# First, watch the naive engine fail: a raw time-travel read at a BOUNDED
# timestamp resolves to a vacuumed version and dies at read time.
naive_t = logical_dates[3]   # inside the vacuumed window
try:
    dt = DeltaTable(str(ORDERS))
    dt.load_as_version(naive_t)
    dt.to_pyarrow_table()
except FileNotFoundError as e:
    print(f"naive AS OF {naive_t.strftime('%H:%M:%S')}: FileNotFoundError")
    print("  ← the log resolved the timestamp to a vacuumed version and the")
    print("    read exploded at runtime. No plan-time warning. This is the lie.\n")

# The gated loop: skip UNACHIEVABLE, clamp BOUNDED to the boundary (the
# earliest surviving state — the best available evidence), stamp everything.
boundary_version = wm.boundary["version"]
partitions, grid = [], []

for i, logical_date in enumerate(logical_dates):
    zone = report.query(logical_date)
    label = f"run {i}  ({logical_date.strftime('%H:%M:%S')})"

    if zone.status == PitStatus.UNACHIEVABLE:
        # AirflowSkipException in the real DAG — never fabricate
        grid.append((label, "SKIPPED", f"limiting: {zone.limiting_chains}"))
        continue

    dt = DeltaTable(str(ORDERS))
    if zone.status == PitStatus.BOUNDED:
        # State at logical_date is destroyed; the boundary is the best
        # available evidence. Serve it clamped — and stamped.
        dt.load_as_version(boundary_version)
        detail = f"clamped to boundary v{boundary_version}"
    else:
        dt.load_as_version(logical_date)   # real timestamp time travel
        detail = f"read v{dt.version()}"
    df = dt.to_pyarrow_table().to_pandas()

    part = (df.groupby("customer_id", as_index=False)["amount"].sum()
              .assign(ds=logical_date.strftime("%H:%M:%S"),
                      pit_verdict=zone.status.value))
    partitions.append(part)
    grid.append((label, zone.status.value, f"{detail}, {len(df)} rows"))

print(f"{'logical date':<22} {'outcome':<14} detail")
print("─" * 64)
for label, status, detail in grid:
    print(f"{label:<22} {status:<14} {detail}")

naive AS OF 15:48:51: FileNotFoundError
  ← the log resolved the timestamp to a vacuumed version and the
    read exploded at runtime. No plan-time warning. This is the lie.

logical date           outcome        detail
────────────────────────────────────────────────────────────────
run 0  (15:48:28)      SKIPPED        limiting: ['delta://orders']
run 1  (15:48:38)      SKIPPED        limiting: ['delta://orders']
run 2  (15:48:49)      BOUNDED        clamped to boundary v5, 6 rows
run 3  (15:48:51)      BOUNDED        clamped to boundary v5, 6 rows
run 4  (15:48:53)      BOUNDED        clamped to boundary v5, 6 rows
run 5  (15:48:55)      BOUNDED        clamped to boundary v5, 6 rows
run 6  (15:48:55)      CERTAIN        read v5, 6 rows
run 7  (15:48:57)      CERTAIN        read v7, 6 rows
run 8  (15:48:59)      CERTAIN        read v7, 6 rows


**Reading the grid:** the two pre-creation dates were *skipped*, not
zero-filled. The BOUNDED dates could not be read at their requested
timestamps at all — the naive attempt above proved that — so the loop
served the **boundary state clamped and stamped**: real values, honestly
labelled as "the earliest surviving evidence", never passed off as the
state at the requested time. The CERTAIN dates time-travelled exactly.

In [5]:
reconstructed = pd.concat(partitions, ignore_index=True)
print("reporting.revenue_daily — reconstructed, with epistemic status:")
print(reconstructed.to_string(index=False))

n_bounded = (reconstructed.pit_verdict == "BOUNDED").sum()
n_certain = (reconstructed.pit_verdict == "CERTAIN").sum()
print(f"\n{n_certain} CERTAIN rows, {n_bounded} BOUNDED rows, "
      f"{sum(1 for _, s, _ in grid if s == 'SKIPPED')} partitions honestly absent")

reporting.revenue_daily — reconstructed, with epistemic status:
customer_id  amount       ds pit_verdict
         C0  1203.0 15:48:49     BOUNDED
         C1  1205.0 15:48:49     BOUNDED
         C2  1207.0 15:48:49     BOUNDED
         C0  1203.0 15:48:51     BOUNDED
         C1  1205.0 15:48:51     BOUNDED
         C2  1207.0 15:48:51     BOUNDED
         C0  1203.0 15:48:53     BOUNDED
         C1  1205.0 15:48:53     BOUNDED
         C2  1207.0 15:48:53     BOUNDED
         C0  1203.0 15:48:55     BOUNDED
         C1  1205.0 15:48:55     BOUNDED
         C2  1207.0 15:48:55     BOUNDED
         C0  1203.0 15:48:55     CERTAIN
         C1  1205.0 15:48:55     CERTAIN
         C2  1207.0 15:48:55     CERTAIN
         C0  1203.0 15:48:57     CERTAIN
         C1  1205.0 15:48:57     CERTAIN
         C2  1207.0 15:48:57     CERTAIN
         C0  1203.0 15:48:59     CERTAIN
         C1  1205.0 15:48:59     CERTAIN
         C2  1207.0 15:48:59     CERTAIN

9 CERTAIN rows, 12 BOUNDED rows, 

## 4. The production DAG

The same loop, expressed as an Airflow DAG. Also shipped as
[`examples/airflow_reconstruct_dag.py`](../examples/airflow_reconstruct_dag.py).

```python
from airflow.decorators import dag, task
from airflow.exceptions import AirflowSkipException
from datetime import datetime
import alethe
from alethe import PitStatus
from alethe.integrations import DbtLineage

@dag(schedule="@daily", start_date=datetime(2024, 1, 1), catchup=True)
def reconstruct_revenue():

    @task
    def rebuild_partition(logical_date=None):
        chains  = alethe.load_watermarks("s3://audit/watermarks.jsonl")
        lineage = DbtLineage("target/manifest.json")
        wms     = lineage.resolve_watermarks("revenue_summary", chains)
        report  = lineage.pit_report("revenue_summary", watermarks=wms)

        zone = report.query(logical_date)
        if zone.status == PitStatus.UNACHIEVABLE:
            raise AirflowSkipException(
                f"{logical_date}: {zone.limiting_chains} did not exist")

        res = lineage.rewrite_model(
            "revenue_summary", logical_date,
            watermarks=wms,
            compiled_path="target/compiled/analytics",
            dialect="spark",
        )
        spark.sql(f\"\"\"
            INSERT OVERWRITE reporting.revenue_daily
            PARTITION (ds = '{logical_date:%Y-%m-%d}')
            SELECT *, '{zone.status.value}' AS pit_verdict
            FROM ({res.sql})
        \"\"\")

    rebuild_partition()

reconstruct_revenue()
```

Pair it with a second scheduled DAG that runs `alethe.watermark()` +
`alethe.record()` daily — the **witness heartbeat**. Beyond keeping the
manifest fresh, it detects boundary *rewinds* (a watermark moving
backwards means the manifest or the table was tampered with — watermarks
are monotone by spec §4).